In [0]:
# Databricks notebook source
# Gold Layer - dim_product (SCD Type 1)
# Same pattern as dim_customer: batch MERGE, run before fact_sales.

from pyspark.sql import functions as F
from pyspark.sql.window import Window

# COMMAND ----------

spark.sql(
    """
    CREATE TABLE IF NOT EXISTS retail_sales_dev.gold.dim_product (
        ProductSK BIGINT GENERATED ALWAYS AS IDENTITY,
        StockCode STRING,
        Description STRING,
        CreatedTimestamp TIMESTAMP,
        UpdatedTimestamp TIMESTAMP
    ) USING DELTA
    """
)


In [0]:

# COMMAND ----------

silver = spark.table("retail_sales_dev.silver.sales_cleaned")

# Description text varies slightly across invoices for the same StockCode
# (casing, trailing text). Resolve to the most frequent Description per code.
desc_counts = silver.groupBy("StockCode", "Description").agg(F.count("*").alias("cnt"))

w = Window.partitionBy("StockCode").orderBy(F.desc("cnt"))


In [0]:

product_source = (
    desc_counts
    .withColumn("rn", F.row_number().over(w))
    .filter("rn = 1")
    .select("StockCode", "Description")
    .withColumn("UpdatedTimestamp", F.current_timestamp())
)

product_source.createOrReplaceTempView("product_source")

# COMMAND ----------

spark.sql(
    """
    MERGE INTO retail_sales_dev.gold.dim_product AS target
    USING product_source AS source
    ON target.StockCode = source.StockCode
    WHEN MATCHED AND target.Description <> source.Description THEN
        UPDATE SET
            target.Description = source.Description,
            target.UpdatedTimestamp = source.UpdatedTimestamp
    WHEN NOT MATCHED THEN
        INSERT (StockCode, Description, CreatedTimestamp, UpdatedTimestamp)
        VALUES (source.StockCode, source.Description, current_timestamp(), source.UpdatedTimestamp)
    """
)

display(spark.table("retail_sales_dev.gold.dim_product").orderBy("ProductSK"))